In [16]:
import deeplabcut
import os
import numpy as np

from deeplabcut.modelzoo import build_weight_init
from modules.dlc_utils import load_config

from modules.train_utils import delete_created_training_artifacts

project_path = 'projects/rat'
config_path = os.path.join(project_path, "config.yaml")
project_config = load_config(project_path)

In [17]:
# 1: 1253 images and 66 for testing: Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
# 2. Using 1343 images and 71 for testing: Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
# 3. Using 1724 images and 91 for testing: Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
# 6. Using 1591 images and 84 for testing: Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
# 7. Using 1591 images and 84 for testing: Trained with superanimal_quadruped rtmpose_x + top-down detector + unfreeze bn stats + default detector training epochs
shuffle = 7

# initialize model weight

In [ ]:
# superanimal_name = 'superanimal_mouse'
superanimal_name = 'superanimal_humanbody'
model_name = "rtmpose_x"
# detector_name="fasterrcnn_resnet50_fpn_v2"
detector_name = None
customized_detector_checkpoint="projects/rat/dlc-models-pytorch/iteration-0/ratOct2-trainset95shuffle6/train/snapshot-detector-best-190.pt"
weight_init = build_weight_init(
            cfg = config_path,
            super_animal= superanimal_name,
            model_name=model_name,
            detector_name=detector_name,
            customized_detector_checkpoint=customized_detector_checkpoint,
            with_decoder=False
)


NotImplementedError: Weight Initialization, Transfer-Learning and Finetuning is currently not supported for superanimal_humanbody

# Create training data

In [ ]:

np.random.seed(42)
delete_created_training_artifacts(project_path, shuffle=shuffle, iteration=0)
    
dt = deeplabcut.create_training_dataset(
    config_path, 
    Shuffles=[shuffle],    
    weight_init=weight_init, 
    net_type=model_name,  
    detector_type=detector_name,
    userfeedback=False)

Removed 0 dataset entries, 0 metadata entries, and 0 model directories for shuffle6.


# replace data augmentation parameters

In [9]:
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch
import yaml

loader = dlc_torch.DLCLoader(
    config=config_path,  
    trainset_index=0,
    shuffle=shuffle,
)

# Get the pytorch config
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"
model_cfg = read_config_as_dict(pytorch_config_path)

In [10]:
# Set freeze_bn_stats=False for GPU training with large batch size
model_cfg["detector"]["model"]["freeze_bn_stats"] = False
# model_cfg["detector"]["train_settings"]["batch_size"] = 4

dlc_torch.config.write_config(pytorch_config_path, model_cfg)

# Add augmentation

In [11]:
{i: project_config['bodyparts'][i] for i in range(len(project_config['bodyparts']))}

{0: 'head',
 1: 'nose',
 2: 'spine1',
 3: 'spine2',
 4: 'spine3',
 5: 'tailbase',
 6: 'tail1',
 7: 'tail2',
 8: 'tail_tip',
 9: 'L_hip',
 10: 'L_backpaw',
 11: 'R_backpaw',
 12: 'L_shoulder',
 13: 'R_frontpaw',
 14: 'R_shoulder',
 15: 'R_hip',
 16: 'R_knee',
 17: 'L_knee',
 18: 'L_frontpaw'}

In [12]:
# Symmetric pairs (left-right): 
# L_hip(9) ↔ R_hip(15), L_backpaw(10) ↔ R_backpaw(11), 
# L_shoulder(12) ↔ R_shoulder(14), L_frontpaw(18) ↔ R_frontpaw(13), 
# L_knee(17) ↔ R_knee(16)

model_cfg["data"]["train"]["hflip"] = {
    "p": 0.5,  # 50% probability
    "symmetries": [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
}

model_cfg["data"]["train"]['covering'] = True
model_cfg["data"]["train"]['grayscale'] = True
model_cfg["data"]["train"]['gaussian_noise'] = 20

dlc_torch.config.write_config(pytorch_config_path, model_cfg)
print(model_cfg["data"]["train"]["hflip"])

{'p': 0.5, 'symmetries': [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]}


# Train

In [13]:
# delete all pt files 
import glob
pt_files = glob.glob(f'projects/rat_pose/dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}/train/*.pt')
for f in pt_files:
    os.remove(f)

In [15]:
deeplabcut.train_network(
    config_path,
    shuffle=shuffle,
    epochs=400,
    save_epochs=10,
    detector_epochs = 200,
    superanimal_name=superanimal_name,
    batch_size= 16,
    keepdeconvweights=False,
    device="cuda:0",
    superanimal_transfer_learning=True
    )

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    gaussian_noise: 20
    motion_blur: True
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
    random_bbox_transform:
      shift_factor: 0.16
      shift_prob: 0.3
      scale_factor: [0.75, 1.25]
      scale_prob: 1.0
      p: 1.0
    hflip:
      p: 0.5
      symmetries: [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
    covering: True
    grayscale: True
detector:
  data:
    colormode: RGB
    inference:
      normalize_images: True
    train:
      affine:
        p: 0.5
        rotation: 30
        scaling: [1.0, 1.0]
        translation: 40
      collate:
        type: ResizeFromDataSizeCollate
        min_scale: 0.4
        max_scale: 1.0
        min_short_side: 128
        max_sh

In [ ]:
# deeplabcut.train_network(
#     config_path,
#     shuffle=shuffle,
#     epochs=400,
#     save_epochs=10,
#     detector_epochs = 200,
#     superanimal_name=superanimal_name,
#     batch_size= 16,
#     keepdeconvweights=False,
#     device="cuda:0",
#     superanimal_transfer_learning=True,
#     snapshot_path = "projects/rat/dlc-models-pytorch/iteration-0/ratOct2-trainset95shuffle3/train/snapshot-best-060.pt",
#     detector_path  = "projects/rat/dlc-models-pytorch/iteration-0/ratOct2-trainset95shuffle3/train/snapshot-detector-best-080.pt"
#     )

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
    random_bbox_transform:
      shift_factor: 0.16
      shift_prob: 0.3
      scale_factor: [0.75, 1.25]
      scale_prob: 1.0
      p: 1.0
    hflip:
      p: 0.5
      symmetries: [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
detector:
  data:
    colormode: RGB
    inference:
      normalize_images: True
    train:
      affine:
        p: 0.5
        rotation: 30
        scaling: [1.0, 1.0]
        translation: 40
      collate:
        type: ResizeFromDataSizeCollate
        min_scale: 0.4
        max_scale: 1.0
        min_short_side: 128
        max_short_side: 1152
        multiple_of: 